# Rekurrente nevrale nettverk

I forrige modul dekket vi rike semantiske representasjoner av tekst. Arkitekturen vi har brukt fanger opp den samlede betydningen av ordene i en setning, men tar ikke hensyn til **rekkefølgen** av ordene, fordi aggregeringsoperasjonen som følger etter innebyggingene fjerner denne informasjonen fra den opprinnelige teksten. Fordi disse modellene ikke kan representere ordrekkefølge, kan de ikke løse mer komplekse eller tvetydige oppgaver som tekstgenerering eller spørsmål og svar.

For å fange meningen i en tekstsekvens, skal vi bruke en nevralt nettverksarkitektur kalt **rekurrent nevralt nettverk**, eller RNN. Når vi bruker en RNN, sender vi setningen vår gjennom nettverket ett token av gangen, og nettverket produserer en slags **tilstand**, som vi deretter sender til nettverket igjen sammen med neste token.

![Bilde som viser et eksempel på generering med rekurrent nevralt nettverk.](../../../../../translated_images/no/rnn.27f5c29c53d727b5.webp)

Gitt inndatasekvensen av tokens $X_0,\dots,X_n$, lager RNN-en en sekvens av nevrale nettverksblokker, og trener denne sekvensen ende-til-ende ved hjelp av backpropagation. Hver nettverksblokk tar et par $(X_i,S_i)$ som inndata, og produserer $S_{i+1}$ som resultat. Den endelige tilstanden $S_n$ eller utdata $Y_n$ går inn i en lineær klassifiserer for å produsere resultatet. Alle nettverksblokker deler de samme vektene, og trenes ende-til-ende med én gjennomgang av backpropagation.

> Figuren ovenfor viser et rekurrent nevralt nettverk i utbrettet form (til venstre), og i en mer kompakt rekurrent representasjon (til høyre). Det er viktig å forstå at alle RNN-celler har de samme **delbare vektene**.

Fordi tilstandsvektorer $S_0,\dots,S_n$ sendes gjennom nettverket, kan RNN-en lære sekvensielle avhengigheter mellom ord. For eksempel, når ordet *not* vises et sted i sekvensen, kan den lære å negere visse elementer i tilstandsvektoren.

Inni inneholder hver RNN-celle to vektmatriser: $W_H$ og $W_I$, og bias $b$. Ved hver RNN-steg, gitt inndata $X_i$ og inndatatilstand $S_i$, beregnes utdata som $S_{i+1} = f(W_H\times S_i + W_I\times X_i+b)$, hvor $f$ er en aktiveringsfunksjon (ofte $\tanh$).

> For problemer som tekstgenerering (som vi skal dekke i neste enhet) eller maskinoversettelse ønsker vi også å få en utdataverdi ved hvert RNN-steg. I dette tilfellet finnes det også en annen matrise $W_O$, og utdata beregnes som $Y_i=f(W_O\times S_i+b_O)$.

La oss se hvordan rekurrente nevrale nettverk kan hjelpe oss å klassifisere vårt nyhetsdatasett.

> For sandkassemiljøet må vi kjøre følgende celle for å sikre at nødvendig bibliotek er installert, og at data er forhåndslastet. Hvis du kjører lokalt, kan du hoppe over følgende celle.


In [1]:
import sys
!{sys.executable} -m pip install --quiet tensorflow_datasets==4.4.0
!cd ~ && wget -q -O - https://mslearntensorflowlp.blob.core.windows.net/data/tfds-ag-news.tgz | tar xz

In [2]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

# We are going to be training pretty large models. In order not to face errors, we need
# to set tensorflow option to grow GPU memory allocation when required
physical_devices = tf.config.list_physical_devices('GPU') 
if len(physical_devices)>0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

ds_train, ds_test = tfds.load('ag_news_subset').values()

Når du trener store modeller, kan GPU-minnetildeling bli et problem. Vi må kanskje også eksperimentere med forskjellige minibatch-størrelser, slik at dataene passer inn i GPU-minnet vårt, samtidig som treningen går raskt nok. Hvis du kjører denne koden på din egen GPU-maskin, kan du eksperimentere med å justere minibatch-størrelsen for å få raskere trening.

> **Merk**: Enkelte versjoner av NVidia-drivere er kjent for ikke å frigjøre minnet etter trening av modellen. Vi kjører flere eksempler i denne notatboken, og det kan føre til at minnet blir brukt opp i visse oppsett, spesielt hvis du gjør dine egne eksperimenter som en del av samme notatbok. Hvis du støter på merkelige feil når du begynner å trene modellen, kan det hende du bør starte notatbokens kjernen på nytt.


In [3]:
batch_size = 16
embed_size = 64

## Enkel RNN-klassifiserer

I tilfellet med en enkel RNN er hver rekurrent enhet et enkelt lineært nettverk, som tar inn en inputvektor og en tilstandsvektor, og produserer en ny tilstandsvektor. I Keras kan dette representeres med `SimpleRNN`-laget.

Selv om vi kan sende en-hot kodede token direkte til RNN-laget, er dette ikke en god idé på grunn av deres høye dimensjonalitet. Derfor vil vi bruke et embedding-lag for å senke dimensjonaliteten på ordvektorer, etterfulgt av et RNN-lag, og til slutt en `Dense`-klassifiserer.

> **Merk**: I tilfeller der dimensjonaliteten ikke er så høy, for eksempel ved bruk av tegnnivå-tokenisering, kan det gi mening å sende en-hot kodede token direkte inn i RNN-cellen.


In [4]:
vocab_size = 20000

vectorizer = keras.layers.experimental.preprocessing.TextVectorization(
    max_tokens=vocab_size,
    input_shape=(1,))

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
text_vectorization (TextVect (None, None)              0         
_________________________________________________________________
embedding (Embedding)        (None, None, 64)          1280000   
_________________________________________________________________
simple_rnn (SimpleRNN)       (None, 16)                1296      
_________________________________________________________________
dense (Dense)                (None, 4)                 68        
Total params: 1,281,364
Trainable params: 1,281,364
Non-trainable params: 0
_________________________________________________________________


> **Merk:** Vi bruker et utrent innebyggingslag her for enkelhets skyld, men for bedre resultater kan vi bruke et forhåndstrent innebyggingslag ved hjelp av Word2Vec, som beskrevet i forrige enhet. Det ville være en god øvelse for deg å tilpasse denne koden til å fungere med forhåndstrente innebygginger.

Nå skal vi trene RNN-en vår. RNN-er generelt er ganske vanskelige å trene, fordi når RNN-cellene blir utrullet langs sekvenslengden, er det resulterende antallet lag som er involvert i bakpropagering ganske stort. Derfor må vi velge en mindre læringsrate, og trene nettverket på et større datasett for å oppnå gode resultater. Dette kan ta ganske lang tid, så det er foretrukket å bruke en GPU.

For å få opp farten, vil vi bare trene RNN-modellen på nyhetstitler, og utelate beskrivelsen. Du kan prøve å trene med beskrivelsen og se om du kan få modellen til å lære.


In [5]:
def extract_title(x):
    return x['title']

def tupelize_title(x):
    return (extract_title(x),x['label'])

print('Training vectorizer')
vectorizer.adapt(ds_train.take(2000).map(extract_title))

Training vectorizer


In [6]:
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize_title).batch(batch_size),validation_data=ds_test.map(tupelize_title).batch(batch_size))

7500/7500 [==============================] - 82s 11ms/step - loss: 0.6629 - acc: 0.7623 - val_loss: 0.5559 - val_acc: 0.7995


> **Merk** at nøyaktigheten sannsynligvis er lavere her, fordi vi kun trener på nyhetstitler.


## Gjennomgang av variable sekvenser 

Husk at `TextVectorization` laget automatisk vil fylle på sekvenser med variabel lengde i en minibatch med pad-tokener. Det viser seg at disse tokenene også deltar i treningen, og de kan komplisere modellens konvergens.

Det finnes flere tilnærminger vi kan bruke for å minimere mengden padding. En av dem er å omorganisere datasettet etter sekvenslengde og gruppere alle sekvenser etter størrelse. Dette kan gjøres ved hjelp av `tf.data.experimental.bucket_by_sequence_length` funksjonen (se [dokumentasjon](https://www.tensorflow.org/api_docs/python/tf/data/experimental/bucket_by_sequence_length)). 

En annen tilnærming er å bruke **maskering**. I Keras støtter noen lag tilleggsinput som viser hvilke token som skal tas med i betraktningen under trening. For å inkludere maskering i modellen vår, kan vi enten legge til et eget `Masking` lag ([docs](https://keras.io/api/layers/core_layers/masking/)), eller vi kan spesifisere `mask_zero=True` parameteren for `Embedding` laget vårt.

> **Merk**: Denne treningen vil ta omtrent 5 minutter per epoke på hele datasettet. Du kan når som helst avbryte treningen hvis tålmodigheten tar slutt. Det du også kan gjøre, er å begrense mengden data som brukes til trening, ved å legge til `.take(...)` klausul etter `ds_train` og `ds_test` datasett.


In [7]:
def extract_text(x):
    return x['title']+' '+x['description']

def tupelize(x):
    return (extract_text(x),x['label'])

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size,embed_size,mask_zero=True),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),validation_data=ds_test.map(tupelize).batch(batch_size))

7500/7500 [==============================] - 371s 49ms/step - loss: 0.5401 - acc: 0.8079 - val_loss: 0.3780 - val_acc: 0.8822


Nå som vi bruker maskering, kan vi trene modellen på hele datasettet av titler og beskrivelser.

> **Merk**: Har du lagt merke til at vi har brukt en vektoriserer trent på nyhetstitlene, og ikke hele artikkelteksten? Dette kan potensielt føre til at noen av tokenene blir ignorert, så det er bedre å trene vektorisereren på nytt. Det kan imidlertid bare ha en veldig liten effekt, så vi vil holde oss til den tidligere forhåndstrente vektorisereren for enkelhets skyld.


## LSTM: Lang korttidsminne

Et av hovedproblemene med RNN-er er **forsvinnende gradienter**. RNN-er kan være ganske lange, og kan ha problemer med å propagere gradientene helt tilbake til det første laget i nettverket under bakoverpropagasjon. Når dette skjer, kan ikke nettverket lære relasjoner mellom fjerne token. En måte å unngå dette problemet på er å innføre **eksplisitt tilstandsadministrasjon** ved å bruke **porter**. De to mest vanlige arkitekturene som introduserer porter er **lang korttidsminne** (LSTM) og **gated relay unit** (GRU). Vi skal dekke LSTM her.

![Bilde som viser et eksempel på en lang korttidshukommelsescelle](../../../../../lessons/5-NLP/16-RNN/images/long-short-term-memory-cell.svg)

Et LSTM-nettverk er organisert på en måte som ligner en RNN, men det finnes to tilstander som sendes fra lag til lag: den faktiske tilstanden $c$, og den skjulte vektoren $h$. Ved hver enhet kombineres den skjulte vektoren $h_{t-1}$ med input $x_t$, og sammen kontrollerer de hva som skjer med tilstanden $c_t$ og output $h_{t}$ gjennom **porter**. Hver port har sigmoid-aktivering (output i området $[0,1]$), som kan tenkes som en bitvis maske når den multipliseres med tilstandsvektoren. LSTM-er har følgende porter (fra venstre til høyre på bildet over):
* **glemmeport** som bestemmer hvilke komponenter av vektoren $c_{t-1}$ vi må glemme, og hvilke vi skal føre videre.
* **inngangsport** som bestemmer hvor mye informasjon fra inputvektoren og tidligere skjulte vektor som skal innarbeides i tilstandsvektoren.
* **utgangsport** som tar den nye tilstandsvektoren og avgjør hvilke av dens komponenter som skal brukes til å produsere den nye skjulte vektoren $h_t$.

Komponentene i tilstanden $c$ kan tenkes som flagg som kan slås av og på. For eksempel, når vi møter navnet *Alice* i sekvensen, antar vi at det refererer til en kvinne, og hever flagget i tilstanden som sier at vi har et hunkjønnsnavn i setningen. Når vi videre støter på ordene *og Tom*, vil vi heve flagget som sier at vi har et flertallsnavn. Ved å manipulere tilstand kan vi slik holde oversikt over de grammatiske egenskapene til setningen.

> **Merk**: Her er en flott ressurs for å forstå innsiden av LSTM-er: [Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) av Christopher Olah.

Selv om den interne strukturen til en LSTM-celle kan se kompleks ut, skjuler Keras denne implementasjonen inne i `LSTM` laget, så det eneste vi trenger å gjøre i eksemplet over er å erstatte den rekurrente laget:


In [8]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.LSTM(8),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(8),validation_data=ds_test.map(tupelize).batch(8))

15000/15000 [==============================] - 188s 13ms/step - loss: 0.5692 - acc: 0.7916 - val_loss: 0.3441 - val_acc: 0.8870


> **Merk** at trening av LSTM-er også er ganske treg, og du vil kanskje ikke se mye økning i nøyaktighet i begynnelsen av treningen. Du må kanskje fortsette treningen en stund for å oppnå god nøyaktighet.


## Toveis og flerlags RNN-er

I eksemplene våre så langt opererer de rekurrente nettverkene fra starten av en sekvens til slutten. Dette føles naturlig for oss fordi det følger samme retning som vi leser eller hører tale i. For scenarioer som krever tilfeldig tilgang til inngangssekvensen, gir det imidlertid mer mening å kjøre den rekurrente beregningen i begge retninger. RNN-er som tillater beregninger i begge retninger kalles **toveis** RNN-er, og de kan opprettes ved å pakke det rekurrente laget inn i et spesielt `Bidirectonal`-lag.

> **Merk**: `Bidirectional`-laget lager to kopier av laget inni seg, og setter `go_backwards`-egenskapen til en av kopiene til `True`, slik at den går i motsatt retning langs sekvensen.

Rekurrente nettverk, enten enveis eller toveis, fanger mønstre i en sekvens, og lagrer dem i tilstandsvektorer eller returnerer dem som output. Som med konvolusjonsnettverk, kan vi bygge et nytt rekurrent lag etter det første for å fange høyere nivå-mønstre, bygget fra lavere nivå-mønstre som er ekstrahert av det første laget. Dette fører oss til begrepet **flerlags RNN**, som består av to eller flere rekurrente nettverk, hvor output fra det forrige laget sendes til neste lag som input.

![Bilde som viser et flerlags langt-korttidshukommelse- RNN](../../../../../translated_images/no/multi-layer-lstm.dd975e29bb2a59fe.webp)

*Bilde fra [det fantastiske innlegget](https://towardsdatascience.com/from-a-lstm-cell-to-a-multilayer-lstm-network-with-pytorch-2899eb5696f3) av Fernando López.*

Keras gjør det enkelt å konstruere disse nettverkene, fordi du bare trenger å legge til flere rekurrente lag i modellen. For alle lag bortsett fra det siste må vi angi `return_sequences=True`-parameteren, fordi vi trenger at laget returnerer alle mellomliggende tilstander, og ikke bare den endelige tilstanden i den rekurrente beregningen.

La oss bygge en to-lags toveis LSTM for vårt klassifiseringsproblem.

> **Merk** denne koden tar igjen ganske lang tid å fullføre, men den gir oss den høyeste nøyaktigheten vi har sett så langt. Så det er kanskje verdt å vente og se resultatet.


In [9]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, 128, mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(64,return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(64)),    
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),
          validation_data=ds_test.map(tupelize).batch(batch_size))

5045/7500 [===================>..........] - ETA: 2:33 - loss: 0.3709 - acc: 0.8706

## RNN-er for andre oppgaver

Så langt har vi fokusert på å bruke RNN-er til å klassifisere sekvenser av tekst. Men de kan håndtere mange flere oppgaver, som tekstgenerering og maskinoversettelse &mdash; vi vil se på disse oppgavene i neste enhet.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokumentet er oversatt ved hjelp av AI-oversettelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selv om vi streber etter nøyaktighet, vær oppmerksom på at automatiske oversettelser kan inneholde feil eller unøyaktigheter. Det opprinnelige dokumentet på originalspråket skal betraktes som den autoritative kilden. For kritisk informasjon anbefales profesjonell menneskelig oversettelse. Vi er ikke ansvarlige for eventuelle misforståelser eller feiltolkninger som oppstår ved bruk av denne oversettelsen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
